# Lección 1.4 — Tipos de problema: regresión, clasificación, clustering, ranking y generación

> **Módulo 1 · ¿Qué es aprender para una máquina?** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 65–80 min (lección con ejercicios intercalados ≈ 45 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 1.2 (vocabulario) y 1.3 (tipos de aprendizaje).

### 🎯 Objetivos
Al terminar esta lección podrás:
- Reconocer los cinco tipos de problema según **qué tipo de respuesta** se busca: un número, una categoría, grupos, un orden o contenido nuevo.
- Traducir una pregunta de negocio al tipo de problema de ML adecuado.
- Ver que **la misma pregunta** puede plantearse de varias formas (p. ej., regresión o clasificación) y elegir la más útil.
- Evitar errores típicos, como tratar un código postal como un número.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números o palabras.

## ▶️ Preparación

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

print("✅ Todo listo")

---
## 1. 📖 La intuición: ¿qué forma tiene la respuesta?

En la Lección 1.3 clasificamos según **qué recibe** el modelo para aprender. Ahora miramos **qué tiene que devolver**. Piensa en preguntas que le harías a un asistente en una app de comida a domicilio:

| Pregunta | Forma de la respuesta | Tipo de problema |
|---|---|---|
| "¿En cuántos minutos llega mi pedido?" | Un **número** | **Regresión** |
| "¿Llegará tarde?" / "¿Es comida mexicana, china o italiana?" | Una **categoría** | **Clasificación** |
| "¿Qué tipos de clientes tenemos?" | **Grupos** (sin nombres previos) | **Clustering** (agrupamiento) |
| "¿Qué 5 restaurantes le muestro primero a Ana?" | Un **orden** | **Ranking** |
| "Escribe la descripción de este plato" | **Contenido nuevo** | **Generación** |

Truco para no perderte: **imagina la respuesta antes de pensar en el modelo.**

### ✋ Ejercicio 1 — ¿Qué forma tiene la respuesta?
1. "¿Cuánto venderemos el mes que viene?"
2. "¿Esta reseña es positiva, neutral o negativa?"
3. "¿En qué orden muestro los resultados de búsqueda de 'zapatos rojos'?"
4. "¿Qué grupos de productos se compran juntos?"
5. "Genera una imagen para la portada del menú."

✍️ *Mis respuestas:* 1 → ___ · 2 → ___ · 3 → ___ · 4 → ___ · 5 → ___

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Regresión** (un número: unidades o dinero) · 2. **Clasificación** (3 categorías) · 3. **Ranking** · 4. **Clustering** · 5. **Generación**.
</details>

---
## 2. 📖 Regresión: predecir un número

La respuesta es un **número continuo**: precio, temperatura, minutos, ventas. Vamos a usar el historial de 500 pedidos de una app de comida.

In [ ]:
rng = np.random.default_rng(1)
n = 500
pedidos = pd.DataFrame({
    "distancia_km": np.round(rng.uniform(0.5, 10, n), 1),
    "hora_pico":    rng.integers(0, 2, n),           # 1 = pedido en hora pico
    "platos":       rng.integers(1, 6, n),
})
pedidos["minutos"] = np.round(10 + 3 * pedidos.distancia_km + 12 * pedidos.hora_pico
                              + 2 * pedidos.platos + rng.normal(0, 5, n)).astype(int)
pedidos.head()

### 🔮 Ejercicio 2 — Estima tú primero
Mirando las primeras filas: ¿cuántos minutos crees que tardará un pedido a **5 km**, en **hora pico**, con **2 platos**?

✍️ *Mi estimación:*

In [ ]:
X = pedidos[["distancia_km", "hora_pico", "platos"]]
regresion = LinearRegression().fit(X, pedidos["minutos"])

nuevo = pd.DataFrame({"distancia_km": [5], "hora_pico": [1], "platos": [2]})   # ✏️ cambia el pedido
print(f"Tiempo estimado: {regresion.predict(nuevo)[0]:.1f} minutos")

<details><summary><b>👀 Ver respuesta</b></summary>

El modelo estima **40.3 minutos** (la regla real con que se generaron los datos da 10 + 15 + 12 + 4 = 41; el modelo se acerca mucho). Fíjate en la forma de la respuesta: **un número con decimales**, que puede tomar cualquier valor. Eso es regresión.

> 📝 El nombre "regresión" es histórico (viene de un estudio de 1886 sobre la estatura de padres e hijos). No significa "volver atrás": significa **predecir un número**.
</details>

---
## 3. 📖 Clasificación: predecir una categoría

La respuesta es una **categoría** de un conjunto fijo de opciones:
- **Binaria** (2 opciones): ¿spam o no?, ¿llegará tarde o no?, ¿pagará o no?
- **Multiclase** (más de 2): ¿qué dígito es?, ¿qué especie de flor?, ¿qué tipo de cocina?

Muchos clasificadores, además de la categoría, dan una **probabilidad**, muy útil para decidir (Lección 4.3).

Con los **mismos datos** de pedidos podemos plantear una pregunta de clasificación: **¿llegará tarde?** (más de 45 minutos).

In [ ]:
pedidos["tarde"] = (pedidos["minutos"] > 45).astype(int)
print(f"Pedidos que llegaron tarde: {pedidos.tarde.mean():.0%}")

clasificador = LogisticRegression().fit(X, pedidos["tarde"])
prob = clasificador.predict_proba(nuevo)[0][1]
print(f"¿El pedido nuevo llegará tarde? {'Sí' if clasificador.predict(nuevo)[0] == 1 else 'No'}  (probabilidad: {prob:.0%})")

### ✋ Ejercicio 3 — ¿Qué pregunta es más útil?
Tienes dos modelos para el mismo pedido:
- Regresión: "llegará en **40.3 minutos**".
- Clasificación: "probabilidad de llegar tarde (más de 45 min): **15 %**".

¿Cuál le mostrarías al **cliente**? ¿Cuál usaría el **gerente de operaciones** para decidir si asigna un repartidor extra?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Al **cliente** le sirve más la **regresión**: quiere saber cuándo llega su comida ("unos 40 minutos"). Al **gerente** le puede servir más la **clasificación**: le interesa detectar los pedidos con **riesgo alto de retraso** (por ejemplo, > 60 %) para actuar.

La misma información, planteada como dos tipos de problema distintos, responde a **necesidades distintas**. Elegir el planteamiento es una decisión de negocio (Lección 1.5).
</details>

---
## 4. 📖 Clustering: encontrar grupos

Ya lo viste en la Lección 1.3: no hay etiqueta, el modelo forma **grupos** de ejemplos parecidos. La diferencia con la clasificación es clave:

| | Clasificación | Clustering |
|---|---|---|
| ¿Las categorías existen antes? | **Sí** (spam / no spam) | **No**: el algoritmo las descubre |
| ¿Necesita etiquetas? | Sí (supervisado) | No (no supervisado) |
| ¿Quién pone nombre a los grupos? | Ya lo tienen | **Tú**, después |

### 🔮 Ejercicio 4
Agrupamos los pedidos por `distancia_km` y `minutos` en **3 grupos**. Antes de ejecutar: ¿qué tres tipos de pedido crees que saldrán?

✍️ *Mi predicción:*

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=0)
pedidos["grupo"] = km.fit_predict(pedidos[["distancia_km", "minutos"]])
print(pedidos.groupby("grupo")[["distancia_km", "minutos", "hora_pico"]].mean().round(1)
      .assign(pedidos=pedidos.grupo.value_counts()).sort_values("minutos"))

<details><summary><b>👀 Ver respuesta</b></summary>

Salen tres grupos ordenados por duración: **cercanos y rápidos** (~3 km, ~24 min), **intermedios** (~5 km, ~38 min) y **lejanos y lentos** (~8 km, ~53 min, el 80 % en hora pico). Los nombres los pusimos nosotros mirando los promedios: el algoritmo solo dijo "0, 1, 2".

Fíjate en que K-means **no sabía** nada de la hora pico, pero el grupo lento tiene muchos más pedidos en hora pico: el agrupamiento captó su efecto a través de los minutos.
</details>

---
## 5. 📖 Ranking: predecir un orden

A veces no importa el valor exacto, sino **el orden**: qué resultado va primero en una búsqueda, qué película va arriba en tu pantalla de inicio, qué candidato revisar antes.

Una forma común de hacerlo: un modelo da una **puntuación** a cada opción (por ejemplo, la probabilidad de que el usuario la elija) y luego se **ordenan** de mayor a menor. Lo que se evalúa es si **los primeros puestos** son buenos.

In [ ]:
# Historial: restaurantes que se le mostraron a Ana y si pidió (1) o no (0)
historial = pd.DataFrame({
    "calificacion": [4.8, 3.9, 4.5, 4.1, 3.5, 4.9, 4.0, 3.2, 4.7, 4.3, 3.8, 4.6],
    "distancia_km": [2.0, 1.0, 6.0, 3.5, 0.8, 7.5, 2.5, 4.0, 1.5, 5.0, 3.0, 2.2],
    "pidio":        [1,   0,   0,   0,   0,   0,   1,   0,   1,   0,   0,   1],
})
modelo_ana = LogisticRegression().fit(historial[["calificacion", "distancia_km"]], historial["pidio"])

candidatos = pd.DataFrame({
    "restaurante":  ["Tacos Lupita", "Sushi Zen", "Pizza Nonna", "Wok Express", "La Parrilla", "Veggie Life"],
    "calificacion": [4.7, 4.9, 4.2, 3.6, 4.5, 4.0],
    "distancia_km": [1.2, 6.5, 2.0, 0.5, 3.0, 1.8],
})
candidatos["puntuacion"] = modelo_ana.predict_proba(candidatos[["calificacion", "distancia_km"]])[:, 1].round(2)
candidatos.sort_values("puntuacion", ascending=False).reset_index(drop=True)

### 🔮 Ejercicio 5 — ¿Quién va primero?
Antes de mirar el resultado (¡tápalo!), mira el historial de Ana: ¿qué le importa más, la calificación o la cercanía? Según eso, ¿qué restaurante pondrías **primero** y cuál **último**? ¿Dónde quedará "Sushi Zen", el mejor calificado?

✍️ *Mi orden:*

<details><summary><b>👀 Ver respuesta</b></summary>

Ana pide en sitios **cercanos** (todos sus pedidos están a ≤ 2.5 km) y nunca en los lejanos, aunque tengan 4.9. El ranking queda: **Tacos Lupita** (4.7, a 1.2 km) primero con 0.70, y **Sushi Zen**, pese a tener la mejor calificación (4.9), **último** con 0.11, por estar a 6.5 km.

¿Te sorprendió **Wok Express** en segundo lugar (0.53), con solo 3.6 de calificación? Al modelo le pesó mucho su cercanía (0.5 km). Pero mira el historial: Ana **no** pidió en los dos sitios cercanos con mala nota (3.9 y 3.5). Con solo 12 ejemplos, el modelo no captó del todo que a Ana también le importa la calificación. **Pocos datos → modelo tosco** (y un buen motivo para revisar los rankings con sentido común).

En un ranking lo importante es **el orden**, sobre todo arriba: si Ana ve primero lo que de verdad pediría, el sistema funciona, aunque las puntuaciones no sean probabilidades "exactas".
</details>

---
## 6. 📖 Generación: crear contenido nuevo

La respuesta es **contenido nuevo**: texto, imágenes, audio, código. Ya construiste un generador de refranes en la Lección 0.2. Hagamos uno de **nombres de restaurantes**: aprende qué **letra** suele seguir a cada par de letras en nombres reales, e inventa nombres nuevos.

In [ ]:
nombres = ["la casona", "el rincon", "la cocina", "el fogon", "la terraza", "el portal", "la cantina",
           "el molino", "la parrilla", "el mesón", "la taberna", "el comal", "la cabaña", "el jardin"]

siguientes = {}
for nombre in nombres:
    texto = "^^" + nombre + "$"                  # ^ = inicio, $ = fin
    for i in range(len(texto) - 2):
        siguientes.setdefault(texto[i:i + 2], []).append(texto[i + 2])

def inventar(azar):
    actual, resultado = "^^", ""
    while True:
        letra = azar.choice(siguientes[actual])
        if letra == "$":
            return resultado
        resultado += letra
        actual = actual[1] + letra

azar = random.Random(3)                          # ✏️ cambia la semilla
inventados = [inventar(azar) for _ in range(8)]
print([(n, "✨ nuevo" if n not in nombres else "copiado") for n in inventados])

### ✋ Ejercicio 6
1. ¿Cuántos de los 8 nombres son **nuevos** y cuántos **copiados** de la lista original?
2. ¿Qué tipo de **aprendizaje** (Lección 1.3) usa este generador para conseguir sus "etiquetas"?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con la semilla 3 salen **4 nuevos** (*"la comal"*, *"el jarraza"*, *"la portabaña"*, *"el rino"*) y **4 copiados** ("el fogon" y "el mesón", dos veces cada uno). Copiar a veces los originales es un comportamiento real de los modelos generativos (la **memorización**), y es un problema serio cuando lo copiado tiene derechos de autor o datos privados.
2. **Auto-supervisado**: la "etiqueta" es la letra siguiente del propio texto. Igual que los LLMs, pero con letras en lugar de palabras y mirando solo 2 letras atrás.

> 📝 Fíjate en la diferencia con los otros cuatro tipos: en generación **no hay una única respuesta correcta**. Por eso evaluar modelos generativos es mucho más difícil (Lección 13.9).
</details>

---
## 7. ✋ Ejercicio 7 — Del negocio al tipo de problema

Para cada pregunta de negocio, indica el **tipo de problema** y la **forma de la respuesta**:

| # | Pregunta de negocio |
|---|---|
| 1 | Un hospital quiere estimar cuántos **días** estará internado un paciente |
| 2 | Un banco quiere saber si una transacción es **fraude** |
| 3 | Una tienda de ropa quiere descubrir **estilos** de clientes sin ideas previas |
| 4 | Un portal de empleo quiere ordenar las **ofertas** para cada candidato |
| 5 | Una inmobiliaria quiere escribir automáticamente la **descripción** de cada anuncio |
| 6 | Una fábrica quiere saber si una pieza tiene defecto de tipo **A, B, C o ninguno** |
| 7 | Una aerolínea quiere predecir el **precio óptimo** de un billete |
| 8 | Un hotel quiere predecir la **calificación** (1 a 5 estrellas) que dejará un huésped |

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Regresión** (número de días). 2. **Clasificación binaria**. 3. **Clustering**. 4. **Ranking**. 5. **Generación**. 6. **Clasificación multiclase** (4 categorías). 7. **Regresión**.
8. **¡Depende!** Se puede plantear como **clasificación multiclase** (5 categorías) o como **regresión** (un número entre 1 y 5, p. ej. 4.3). La regresión aprovecha que las estrellas tienen **orden** (equivocarse entre 4 y 5 es menos grave que entre 1 y 5); la clasificación, no. Hay incluso un tipo intermedio llamado **regresión ordinal**. No hay una única respuesta: hay que argumentar.
</details>

---
## 8. 🚨 Errores comunes y señales de alerta

| 🚨 Error | Ejemplo | Qué hacer |
|---|---|---|
| Tratar como número algo que es una **categoría** | Código postal 28001 > 08001 no significa nada | Trátalo como categoría (Lección 2.3) |
| Tratar como categorías sin orden algo que **sí tiene orden** | Estrellas 1–5, talla S/M/L | Considera regresión o codificación ordinal |
| Confundir clasificación con clustering | "Quiero clasificar clientes" sin tener etiquetas | Si no hay categorías previas, es clustering |
| Usar métricas del tipo equivocado | Medir una regresión con "% de aciertos" exactos | Regresión: error medio (MAE, Lección 4.1). Clasificación: precisión, recall… (Lección 4.2) |
| Plantear la pregunta que es fácil y no la que es **útil** | Predecir minutos exactos cuando el gerente solo necesita "¿riesgo de retraso?" | Pregunta antes qué **decisión** se tomará con la predicción |

### 🔮 Ejercicio 8 — El código postal
Un analista entrena una regresión para predecir el precio de casas y mete `codigo_postal` como número (por ejemplo, 28001, 28045, 08001). ¿Qué crees que "aprenderá" el modelo sobre el código postal? ¿Por qué es un problema?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

La regresión aprenderá algo como "cada unidad más de código postal suma (o resta) X pesos", como si 28045 fuera "más" que 28001. Pero los códigos postales son **etiquetas de zona**, no cantidades: 28001 y 28002 pueden ser barrios muy distintos, y 08001 no es "menos" que 28001. El modelo aprenderá una relación **sin sentido**.

Solución: tratarlo como **categoría** (one-hot, Lección 2.3) o, mejor aún, sustituirlo por características con significado (precio medio de la zona, distancia al centro…; Lección 2.4).
</details>

---
## 9. 🧠 Resumen en 5 puntos
1. El tipo de problema lo define **la forma de la respuesta**: número, categoría, grupos, orden o contenido nuevo.
2. **Regresión** → número. **Clasificación** → categoría (binaria o multiclase), a menudo con probabilidad.
3. **Clustering** → grupos que no existían antes (sin etiquetas). **Ranking** → orden, importan sobre todo los primeros puestos.
4. **Generación** → contenido nuevo; no hay una única respuesta correcta, y por eso es difícil de evaluar.
5. Una misma pregunta puede plantearse de varias formas: elige la que sirve a **la decisión** que se tomará.

---
## 10. 🏋️ Ejercicios finales

### 10.1 Repaso rápido (opción múltiple)

**1.** Predecir la temperatura de mañana en grados es un problema de…
- a) Clasificación · b) Regresión · c) Clustering · d) Ranking

**2.** "¿Este tumor es benigno o maligno?" es…
- a) Regresión · b) Clasificación binaria · c) Clasificación multiclase · d) Generación

**3.** ¿Cuál es la diferencia **clave** entre clasificación y clustering?
- a) El clustering es más exacto · b) En clasificación las categorías existen antes y hay etiquetas; en clustering se descubren · c) No hay diferencia · d) El clustering solo funciona con imágenes

**4.** En un buscador, ¿qué importa más?
- a) La puntuación exacta de cada resultado · b) Que los primeros resultados sean los mejores · c) Que haya muchos resultados · d) Que los resultados estén en orden alfabético

**5.** ¿Por qué es difícil evaluar un modelo generativo?
- a) Porque es muy lento · b) Porque suele haber muchas respuestas válidas, no una sola correcta · c) Porque no tiene parámetros · d) No es difícil

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** · 2. **b)** · 3. **b)** · 4. **b)** · 5. **b)**
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Una empresa de suscripciones quiere reducir cancelaciones. Plantea el problema de **tres formas distintas** (con tipos de problema distintos). ¿Cuál elegirías para empezar y por qué?

**B.** Un sistema de recomendación de canciones podría plantearse como clasificación ("¿le gustará esta canción: sí o no?") o como ranking ("ordena estas 1,000 canciones"). ¿Qué ventaja tiene pensarlo como ranking?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**A.** Por ejemplo:
- **Clasificación:** ¿este cliente cancelará en los próximos 30 días? (sí/no, con probabilidad).
- **Regresión:** ¿cuántos meses más seguirá suscrito?
- **Clustering:** ¿qué tipos de clientes tenemos y cuáles se parecen a los que cancelan?
Un buen punto de partida suele ser la **clasificación**: permite actuar (ofrecer un descuento a quien tiene riesgo alto) y es más sencilla de evaluar. El clustering puede ayudar a entender **por qué** cancelan.

**B.** En el ranking lo que importa es **qué va arriba**: el usuario solo verá unas pocas canciones. Un clasificador puede decir "sí, le gustará" a 300 canciones, pero no dice cuáles mostrar primero. Pensarlo como ranking hace que el modelo se evalúe (y se optimice) justo por lo que importa: **los primeros puestos**.
</details>

### 10.3 🧩 Práctica integradora: el panel de la app de reparto 🛵

Con los datos de `pedidos`, vas a construir las tres piezas del panel del gerente, cada una de un tipo de problema distinto. 4 pasos cortos; revisa cada respuesta antes de seguir.

**Paso 1 ▶️ — Separa un examen.** Guardamos 100 pedidos que los modelos **no verán** al entrenar (Lección 2.5).

In [ ]:
X = pedidos[["distancia_km", "hora_pico", "platos"]]
X_ent, X_exa, min_ent, min_exa, tarde_ent, tarde_exa = train_test_split(
    X, pedidos["minutos"], pedidos["tarde"], test_size=100, random_state=0)
print(f"Entrenamiento: {len(X_ent)} pedidos · Examen: {len(X_exa)} pedidos")

**Paso 2 🔮 — Regresión: ¿cuántos minutos?** Vamos a medir el **error medio** en minutos y a compararlo con una estrategia "tonta": decir siempre el promedio del entrenamiento. Antes de ejecutar: ¿cuántos minutos crees que se equivoca cada uno, en promedio?

✍️ *Mi predicción:*

In [ ]:
reg = LinearRegression().fit(X_ent, min_ent)
error_modelo = np.mean(abs(reg.predict(X_exa) - min_exa))
error_tonto = np.mean(abs(min_ent.mean() - min_exa))
print(f"Error medio del modelo          : {error_modelo:.1f} minutos")
print(f"Error medio de 'decir el promedio': {error_tonto:.1f} minutos")

<details><summary><b>👀 Ver respuesta</b></summary>

El modelo se equivoca **4.5 minutos** en promedio; decir siempre el promedio, **9.8 minutos**. El modelo reduce el error a menos de la mitad. Comparar con una estrategia tonta (**línea base**) es imprescindible para saber si un modelo aporta algo (Lección 4.7). (El error de ~4.5 minutos se debe sobre todo al ruido: imprevistos que ningún modelo puede adivinar.)
</details>

**Paso 3 ✏️ — Clasificación: ¿llegará tarde?** El gerente quiere una alerta cuando la probabilidad de llegar tarde supere un **umbral**. Prueba `UMBRAL` = 0.5, 0.3 y 0.7 y mira cuántos pedidos tardíos detecta y cuántas falsas alarmas da.

In [ ]:
clf = LogisticRegression().fit(X_ent, tarde_ent)
UMBRAL = 0.5                                     # ✏️ prueba 0.3 y 0.7

alerta = clf.predict_proba(X_exa)[:, 1] >= UMBRAL
print(f"Pedidos tardíos en el examen  : {tarde_exa.sum()}")
print(f"Tardíos detectados por alerta : {(alerta & (tarde_exa == 1)).sum()}")
print(f"Falsas alarmas                : {(alerta & (tarde_exa == 0)).sum()}")

<details><summary><b>👀 Ver respuesta</b></summary>

De **31** pedidos tardíos:
- `UMBRAL = 0.5` → detecta **25**, con **8** falsas alarmas.
- `UMBRAL = 0.3` → detecta **28**, pero con **13** falsas alarmas.
- `UMBRAL = 0.7` → detecta solo **17**, con **0** falsas alarmas.

Bajar el umbral detecta más retrasos a costa de más falsas alarmas; subirlo, al revés. Qué umbral elegir depende de **cuánto cuesta cada error** al negocio (Lecciones 4.3 y 4.8).
</details>

**Paso 4 ✋ — Ranking: ¿a qué pedidos atiendo primero?** El gerente solo puede reforzar **5** pedidos con un repartidor extra. Usa las probabilidades del clasificador para ordenar los pedidos del examen y mira cuántos de los 5 primeros llegaron tarde de verdad.

In [ ]:
ranking = X_exa.assign(prob_tarde=clf.predict_proba(X_exa)[:, 1].round(2), llego_tarde=tarde_exa)
top5 = ranking.sort_values("prob_tarde", ascending=False).head(5)
print(top5)
print(f"\nDe los 5 primeros del ranking, llegaron tarde: {top5.llego_tarde.sum()}")
print(f"Si eligiera 5 al azar, esperaría unos: {5 * tarde_exa.mean():.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Los **5 primeros** del ranking llegaron tarde **los 5**: son pedidos lejanos (8–10 km) en hora pico. Eligiendo al azar, se esperarían solo ~1.6.

**Conclusión del panel:** con **los mismos datos** construiste tres herramientas: un **número** para el cliente (regresión), una **alerta** para el gerente (clasificación) y una **lista priorizada** de dónde actuar (ranking). El tipo de problema lo elige **la decisión que quieres tomar**.
</details>

---
## 📚 Para profundizar
- 📘 **scikit-learn:** el "mapa" *Choosing the right estimator* (busca "scikit-learn algorithm cheat sheet"): un diagrama que empieza preguntando qué tipo de problema tienes.
- 📘 **Google:** *Machine Learning Crash Course*, sección *Framing* (cómo plantear un problema de ML).
- 📺 **Video:** StatQuest, "Machine Learning Fundamentals" (inglés, con subtítulos).

---
⬅️ **Anterior:** [1.3 Tipos de aprendizaje: supervisado, no supervisado, semi-supervisado, auto-supervisado y por refuerzo](1.3-tipos-de-aprendizaje.ipynb) · ➡️ **Siguiente:** [1.5 El ciclo de vida de un proyecto de ML: de la pregunta de negocio al modelo en producción](1.5-ciclo-de-vida-de-un-proyecto.ipynb)